# 01 记忆审计 (Memorization Audit)

借鉴 Profit Mirage 的 FinLeak-Bench，测试 DeepSeek 对 A 股历史事实的记忆程度。

四类探针：
- 价格查询 (price_query)
- 趋势预测 (trend_prediction)
- 事件影响 (event_impact)
- 市场表现 (market_performance)

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_memorization_probes
from src.prompts import memorization_probe_prompt
from src.display_utils import show_probe_result, show_llm_response
import json
import pandas as pd

set_seed()

## 1. 加载记忆探针

In [ ]:
probes = load_memorization_probes()
print(f"Loaded {len(probes)} memorization probes")
for pt in ProbeType:
    count = sum(1 for p in probes if p.probe_type == pt.value)
    print(f"  {pt.value}: {count}")

## 2. 执行记忆探针

In [ ]:
client = LLMClient()

# 收集所有 probe prompts，并发调用
prompts = [memorization_probe_prompt(probe.question) for probe in probes]
responses = client.batch_chat_concurrent(prompts)

results = []
for probe, response in zip(probes, responses):
    results.append({
        "probe_id": probe.id,
        "probe_type": probe.probe_type,
        "question": probe.question,
        "ground_truth": probe.ground_truth,
        "tolerance": probe.tolerance,
        "model_answer": response.raw_response,
    })
    show_llm_response(f"[{probe.id}] {probe.question[:60]}", response.raw_response)

## 3. 人工评估辅助

由于记忆探针的答案需要语义匹配，这里提供评估框架。

评分标准：
- **价格查询**: 在容差范围内 = 1分，方向正确但超出容差 = 0.5分
- **趋势预测**: 方向+幅度正确 = 1分，仅方向正确 = 0.5分
- **事件影响**: 定性完全对齐 = 1分，部分对齐 = 0.5分
- **市场表现**: 板块/标的正确 = 1分，相关但不精确 = 0.5分

In [ ]:
# Use LLM-as-judge to auto-evaluate memorization accuracy (并发)
judge_system = """你是一个评估专家。请判断模型回答与标准答案的匹配程度。
评分标准：
- 1.0: 完全匹配（价格在容差内/方向和幅度正确/定性完全对齐）
- 0.5: 部分匹配（方向正确但细节有误）
- 0.0: 不匹配或拒绝回答
输出JSON：{"score": <0/0.5/1.0>, "reason": "..."}"""

judge_prompts = [
    (judge_system, f"问题：{r['question']}\n标准答案：{r['ground_truth']}（容差：{r['tolerance']}）\n模型回答：{r['model_answer']}")
    for r in results
]
judge_responses = client.batch_chat_concurrent(judge_prompts)

for r, judge_resp in zip(results, judge_responses):
    try:
        judge_data = json.loads(judge_resp.raw_response)
        r["score"] = judge_data.get("score", 0)
        r["judge_reason"] = judge_data.get("reason", "")
    except json.JSONDecodeError:
        r["score"] = None
        r["judge_reason"] = judge_resp.raw_response
    show_probe_result(r["probe_id"], r["question"], r["ground_truth"], r["model_answer"], r["score"])

## 4. 记忆审计结果

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

df = pd.DataFrame(results)
df['score'] = pd.to_numeric(df['score'], errors='coerce')

# Overall accuracy
print(f"Overall memorization score: {df['score'].mean():.2f}")
print(f"Perfect recall rate: {(df['score'] == 1.0).mean():.1%}")
print(f"Partial recall rate: {(df['score'] == 0.5).mean():.1%}")
print(f"No recall rate: {(df['score'] == 0.0).mean():.1%}")

# By probe type
print("\nBy probe type:")
type_scores = df.groupby('probe_type')['score'].agg(['mean', 'count'])
print(type_scores)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart by probe type
type_means = df.groupby('probe_type')['score'].mean()
type_means.plot(kind='bar', ax=axes[0], color=['#2196F3', '#4CAF50', '#FF9800', '#F44336'])
axes[0].set_title('记忆准确率 (按探针类型)')
axes[0].set_ylabel('平均分数')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=45)

# Score distribution
df['score'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#9C27B0')
axes[1].set_title('分数分布')
axes[1].set_xlabel('分数')
axes[1].set_ylabel('数量')

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / 'data' / 'results' / 'memorization_audit.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n结论：")
overall = df['score'].mean()
if overall > 0.7:
    print(f"DeepSeek 对 A 股历史事实的记忆程度较高 (score={overall:.2f})，泄露风险显著")
elif overall > 0.4:
    print(f"DeepSeek 对 A 股历史有中等程度记忆 (score={overall:.2f})，需要进一步反事实测试")
else:
    print(f"DeepSeek 对 A 股历史记忆程度较低 (score={overall:.2f})，但仍需反事实验证")

## 5. 保存结果

In [ ]:
output_path = PROJECT_ROOT / 'data' / 'results' / 'memorization_audit.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Results saved to {output_path}")